# E-commerce Sales Analysis

## Importing Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.options.display.float_format = "{:,.2f}".format

## Loading the Dataset

In [ ]:
df = pd.read_csv("ecommerce_sales_analytics_5000.csv")
df.head()

## Inspect the Data

In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
# Numeric columns
df.describe()

In [ ]:
# Categorical columns
df[["product_category", "region", "payment_method"]].describe()

## Missing Values and Duplicates

In [ ]:
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate order_id:", df["order_id"].duplicated().sum())

In [ ]:
# Nothing is removed if no missing values or duplicates are found
df = df.drop_duplicates().dropna()
print("Shape after cleaning:", df.shape)

## Convert `order_date` to Datetime

In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"], format="%m/%d/%Y")
df["year"] = df["order_date"].dt.year

print(df["order_date"].dtype)
print(df["order_date"].min(), "to", df["order_date"].max())

## Overall Sales and Revenue

In [ ]:
total_revenue = df["revenue"].sum()

print(f"Total revenue:       {total_revenue:,.2f}")
print(f"Total orders:        {len(df):,}")
print(f"Unique customers:    {df['customer_id'].nunique():,}")
print(f"Units sold:          {df['quantity'].sum():,}")
print(f"Average order value: {df['revenue'].mean():,.2f}")

In [ ]:
yearly = df.groupby("year")["revenue"].sum()
monthly = df.set_index("order_date")["revenue"].resample("ME").sum()
monthly = monthly.iloc[:-1]  # drop the last month (only 9 days of data)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(monthly.index, monthly.values, color="steelblue")
axes[0].set(title="Monthly Revenue", xlabel="", ylabel="Revenue")

sns.barplot(x=yearly.index, y=yearly.values, ax=axes[1], color="steelblue")
axes[1].set(title="Yearly Revenue (2035 is a partial year)", xlabel="", ylabel="Revenue")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## Helper: Group Summary
Same summary table reused for category, region and payment method.

In [ ]:
def group_summary(col):
    return df.groupby(col).agg(
        orders=("order_id", "count"),
        total_revenue=("revenue", "sum"),
        avg_order_value=("revenue", "mean"),
        avg_discount=("discount", "mean"),
        avg_delivery_days=("delivery_days", "mean"),
        avg_rating=("customer_rating", "mean"),
    ).sort_values("total_revenue", ascending=False)

## Sales by Product Category

In [ ]:
category = group_summary("product_category")
category

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.barplot(x=category.index, y=category["total_revenue"], ax=axes[0], color="steelblue")
axes[0].set(title="Total Revenue by Category", xlabel="", ylabel="Revenue")

sns.barplot(x=category.index, y=category["avg_order_value"], ax=axes[1], color="seagreen")
axes[1].set(title="Average Order Value by Category", xlabel="", ylabel="Avg order value")

plt.tight_layout()
plt.show()

## Sales by Region

In [ ]:
region = group_summary("region")
region

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.barplot(x=region.index, y=region["total_revenue"], ax=axes[0], color="steelblue")
axes[0].set(title="Total Revenue by Region", xlabel="", ylabel="Revenue")

sns.barplot(x=region.index, y=region["avg_order_value"], ax=axes[1], color="seagreen")
axes[1].set(title="Average Order Value by Region", xlabel="", ylabel="Avg order value")

plt.tight_layout()
plt.show()

In [ ]:
# Revenue by category and region
cat_region = df.pivot_table(index="product_category", columns="region", values="revenue", aggfunc="sum")

plt.figure(figsize=(7, 4))
sns.heatmap(cat_region, annot=True, fmt=",.0f", cmap="Blues")
plt.title("Revenue by Category and Region")
plt.xlabel("")
plt.ylabel("")
plt.show()

## Payment Methods

In [ ]:
payment = group_summary("payment_method")
payment

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.barplot(x=payment.index, y=payment["orders"], ax=axes[0], color="steelblue")
axes[0].set(title="Orders by Payment Method", xlabel="", ylabel="Orders")

sns.barplot(x=payment.index, y=payment["total_revenue"], ax=axes[1], color="seagreen")
axes[1].set(title="Total Revenue by Payment Method", xlabel="", ylabel="Revenue")

plt.tight_layout()
plt.show()

## Discounts

In [ ]:
print(df["discount"].describe())

# Total discount given (gross sales minus revenue)
df["discount_amount"] = df["quantity"] * df["unit_price"] - df["revenue"]
print(f"\nTotal discount given: {df['discount_amount'].sum():,.2f}")

In [ ]:
# Group discounts into bands (maximum discount in the data is 35%)
df["discount_band"] = pd.cut(
    df["discount"],
    bins=[0, 0.10, 0.20, 0.30, 0.35],
    labels=["0-10%", "11-20%", "21-30%", "31-35%"],
    include_lowest=True,
)

discount = df.groupby("discount_band", observed=True).agg(
    orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    avg_order_value=("revenue", "mean"),
    avg_quantity=("quantity", "mean"),
    avg_rating=("customer_rating", "mean"),
)
discount

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.barplot(x=discount.index, y=discount["orders"], ax=axes[0], color="steelblue")
axes[0].set(title="Orders by Discount Band", xlabel="Discount", ylabel="Orders")

sns.barplot(x=discount.index, y=discount["avg_order_value"], ax=axes[1], color="seagreen")
axes[1].set(title="Average Order Value by Discount Band", xlabel="Discount", ylabel="Avg order value")

plt.tight_layout()
plt.show()

## Delivery Days and Customer Ratings

In [ ]:
df[["delivery_days", "customer_rating"]].describe()

In [ ]:
# Average delivery time and rating by group
pd.concat({
    "category": category[["avg_delivery_days", "avg_rating"]],
    "region": region[["avg_delivery_days", "avg_rating"]],
    "payment": payment[["avg_delivery_days", "avg_rating"]],
})

In [ ]:
rating_by_delivery = df.groupby("delivery_days")["customer_rating"].mean()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.countplot(x="delivery_days", data=df, ax=axes[0], color="steelblue")
axes[0].set(title="Orders by Delivery Days", xlabel="Delivery days", ylabel="Orders")

sns.histplot(df["customer_rating"], binwidth=0.5, ax=axes[1], color="seagreen")
axes[1].set(title="Customer Rating Distribution", xlabel="Rating", ylabel="Orders")

sns.barplot(x=rating_by_delivery.index, y=rating_by_delivery.values, ax=axes[2], color="orange")
axes[2].set(title="Average Rating by Delivery Days", xlabel="Delivery days", ylabel="Avg rating", ylim=(0, 5))

plt.tight_layout()
plt.show()

## Relationships Between Variables

In [ ]:
num_cols = ["quantity", "unit_price", "discount", "delivery_days", "customer_rating", "revenue"]
corr = df[num_cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Matrix")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.scatterplot(x="unit_price", y="revenue", data=df, ax=axes[0], alpha=0.3, s=12)
axes[0].set(title="Unit Price vs Revenue")

sns.boxplot(x="quantity", y="revenue", data=df, ax=axes[1], color="lightsteelblue")
axes[1].set(title="Revenue by Quantity")

sns.scatterplot(x="discount", y="revenue", data=df, ax=axes[2], alpha=0.3, s=12)
axes[2].set(title="Discount vs Revenue")

plt.tight_layout()
plt.show()

## Key Results

In [ ]:
# Overall
print(f"Total revenue:          {total_revenue:,.2f}")
print(f"Total orders:           {len(df):,}")
print(f"Average order value:    {df['revenue'].mean():,.2f}")
print(f"Average discount:       {df['discount'].mean() * 100:.1f}%")
print(f"Average delivery days:  {df['delivery_days'].mean():.2f}")
print(f"Average rating:         {df['customer_rating'].mean():.2f}")
print(f"Orders rated 4 or more: {(df['customer_rating'] >= 4).mean() * 100:.1f}%")
print(f"Orders rated 2 or less: {(df['customer_rating'] <= 2).mean() * 100:.1f}%")

In [ ]:
# Top group by revenue and by average order value
for name, table in [("Category", category), ("Region", region), ("Payment method", payment)]:
    top = table["total_revenue"].idxmax()
    share = table.loc[top, "total_revenue"] / total_revenue * 100
    print(f"{name}: highest revenue = {top} ({share:.1f}% of total), "
          f"highest avg order value = {table['avg_order_value'].idxmax()}, "
          f"highest avg rating = {table['avg_rating'].idxmax()}")

In [ ]:
# Discounts
print("Average order value by discount band:")
print(discount["avg_order_value"])
print("\nAverage rating by discount band:")
print(discount["avg_rating"])

In [ ]:
# Correlations
print("Correlation with revenue:")
print(corr["revenue"].drop("revenue").to_string(float_format="{:.3f}".format))
print(f"\nDelivery days vs rating: {corr.loc['delivery_days', 'customer_rating']:.3f}")
print(f"Discount vs rating:      {corr.loc['discount', 'customer_rating']:.3f}")